In [0]:
import delta
import sys

sys.path.insert(0, "../lib/")

import utils

In [0]:
catalog = "bronze"
schema = "upsell"

# tablename = "transacao_produto"
# id_field = "IdTransacao"
# timestamp_field = "_extracted_at"

tablename = dbutils.widgets.get("tablename")
id_field = dbutils.widgets.get("id_field")
timestamp_field = dbutils.widgets.get("timestamp_field")

df_schema = utils.import_schema(tablename)

In [0]:
if not utils.table_exists(spark,catalog, schema, tablename):

        print("Tabela não existente, criando...")

        df_full = (spark.read
                        .format("parquet")
                        .schema(df_schema)
                        .load(f"/Volumes/raw/upsell/full_load/{tablename}/"))

        (df_full.coalesce(1)
                .write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(f"{catalog}.{schema}.{tablename}"))
        
else:
        print("Tabela existente, ignorando full-load")

In [0]:
def upsert(df, batch_id):
        session = df.sparkSession

        df.createOrReplaceGlobalTempView(f"view_{tablename}")
        
        query = f'''
        SELECT *
        FROM global_temp.view_{tablename}
        QUALIFY ROW_NUMBER() OVER(PARTITION BY {id_field} ORDER BY {timestamp_field} DESC) = 1
        '''

        df_cdc = session.sql(query)

        # UPSERT
        bronze_table = delta.DeltaTable.forName(session, f"{catalog}.{schema}.{tablename}")
        (bronze_table.alias("b")
                .merge(df_cdc.alias("d"), f"b.{id_field} = d.{id_field}")
                .whenMatchedDelete(condition= "d._operation = 'DELETE'")
                .whenMatchedUpdateAll(condition= "d._operation = 'UPDATE'")
                .whenNotMatchedInsertAll(condition="d._operation = 'INSERT' OR d._operation = 'UPDATE'")
                .execute()
        )

df_stream = (spark.readStream
                        .format("cloudFiles")
                        .option("cloudFiles.format","parquet")
                        .schema(df_schema)
                        .load(f"/Volumes/raw/upsell/cdc/{tablename}/"))

stream = (df_stream.writeStream
                .option("checkpointLocation", f"/Volumes/raw/upsell/cdc/{tablename}/_checkpoints/")
                .foreachBatch(upsert)
                .trigger(availableNow=True))


In [0]:
start = stream.start()